In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout,
    BatchNormalization
)

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D

from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns


In [ ]:
train_dir = "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training"
test_dir  = "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing"

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

test_datagen = ImageDataGenerator(
    rescale=1./255
)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
print(train_data.class_indices)

In [ ]:
images, labels = next(train_data)

plt.figure(figsize=(10,10))

for i in range(9):

    plt.subplot(3,3,i+1)

    plt.imshow(images[i])

    plt.title(np.argmax(labels[i]))

    plt.axis("off")

plt.show()

In [ ]:
from tensorflow.keras.layers import Input

cnn_model = Sequential([

    Input(shape=(224,224,3)),

    Conv2D(32, (3,3), activation='relu'),

    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),

    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),

    MaxPooling2D(2,2),

    Flatten(),

    Dense(128, activation='relu'),

    Dropout(0.5),

    Dense(4, activation='softmax')

])
cnn_model.summary()

In [ ]:
cnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

cnn_history = cnn_model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    callbacks=[early_stop]
)

In [ ]:
plt.plot(cnn_history.history['accuracy'])
plt.plot(cnn_history.history['val_accuracy'])

plt.title("CNN Accuracy")

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.legend(["Train", "Validation"])

plt.show()

In [ ]:
plt.plot(cnn_history.history['loss'])
plt.plot(cnn_history.history['val_loss'])

plt.title("CNN Loss")

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.legend(["Train", "Validation"])

plt.show()

In [ ]:
cnn_loss, cnn_acc = cnn_model.evaluate(test_data)

print("CNN Accuracy:", cnn_acc)

In [ ]:
predictions = cnn_model.predict(test_data)

predictions = np.argmax(predictions, axis=1)

cm = confusion_matrix(
    test_data.classes,
    predictions
)

plt.figure(figsize=(7,7))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.title("Confusion Matrix")

plt.show()

In [ ]:
print(classification_report(
    test_data.classes,
    predictions
))

In [ ]:
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

In [ ]:
x = base_model.output

x = GlobalAveragePooling2D()(x)

x = Dense(128, activation='relu')(x)

x = Dropout(0.5)(x)

predictions = Dense(4, activation='softmax')(x)

resnet_model = Model(
    inputs=base_model.input,
    outputs=predictions
)

resnet_model.summary()

In [ ]:
resnet_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
resnet_history = resnet_model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=[early_stop]
)

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
efficient_base = EfficientNetB0(

    weights='imagenet',

    include_top=False,

    input_shape=(224,224,3)
)

efficient_base.trainable = False

In [ ]:
x = efficient_base.output

x = GlobalAveragePooling2D()(x)

x = Dense(
    128,
    activation='relu'
)(x)

x = Dropout(0.3)(x)

predictions = Dense(
    4,
    activation='softmax'
)(x)

efficient_model = Model(
    inputs=efficient_base.input,
    outputs=predictions
)

In [ ]:
efficient_model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),

    loss='categorical_crossentropy',

    metrics=['accuracy']
)

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(

    monitor='val_loss',

    factor=0.2,

    patience=2,

    min_lr=0.000001
)

In [ ]:
efficient_history = efficient_model.fit(

    train_data,

    validation_data=val_data,

    epochs=15,

    callbacks=[
        early_stop,
        reduce_lr
    ]
)

In [ ]:
from tensorflow.keras.applications import MobileNetV2
mobile_base = MobileNetV2(

    weights='imagenet',

    include_top=False,

    input_shape=(224,224,3)
)

mobile_base.trainable = False

In [ ]:
x = mobile_base.output

x = GlobalAveragePooling2D()(x)

x = Dense(
    128,
    activation='relu'
)(x)

x = Dropout(0.3)(x)

predictions = Dense(
    4,
    activation='softmax'
)(x)

mobile_model = Model(
    inputs=mobile_base.input,
    outputs=predictions
)

In [ ]:
mobile_model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),

    loss='categorical_crossentropy',

    metrics=['accuracy']
)

In [ ]:
mobile_history = mobile_model.fit(

    train_data,

    validation_data=val_data,

    epochs=15,

    callbacks=[
        early_stop,
        reduce_lr
    ]
)

In [ ]:
resnet_loss, resnet_acc = resnet_model.evaluate(test_data)

efficient_loss, efficient_acc = efficient_model.evaluate(test_data)

mobile_loss, mobile_acc = mobile_model.evaluate(test_data)

print("CNN Accuracy:", cnn_acc)

print("ResNet50 Accuracy:", resnet_acc)

print("EfficientNetB0 Accuracy:", efficient_acc)

print("MobileNetV2 Accuracy:", mobile_acc)

In [ ]:
models = [

    'CNN',

    'ResNet50',

    'EfficientNetB0',

    'MobileNetV2'
]

accuracies = [

    cnn_acc,

    resnet_acc,

    efficient_acc,

    mobile_acc
]

plt.figure(figsize=(8,6))

plt.bar(
    models,
    accuracies
)

plt.ylabel("Accuracy")

plt.title("Model Comparison")

plt.show()

In [ ]:
predictions = mobile_model.predict(test_data)

predictions = np.argmax(
    predictions,
    axis=1
)

cm = confusion_matrix(
    test_data.classes,
    predictions
)

class_labels = [

    "Glioma",

    "Meningioma",

    "No Tumor",

    "Pituitary"
]

plt.figure(figsize=(8,8))

sns.heatmap(

    cm,

    annot=True,

    fmt='d',

    cmap='Blues',

    xticklabels=class_labels,

    yticklabels=class_labels
)

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.title("Confusion Matrix")

plt.show()

In [ ]:
print(classification_report(
    test_data.classes,
    predictions
))

In [ ]:
mobile_model.save("best_brain_tumor_model.keras")

print("Best Model Saved Successfully ✅")

In [ ]:
from tensorflow.keras.models import load_model

model = load_model("best_brain_tumor_model.keras")

In [ ]:
from tensorflow.keras.preprocessing import image

img_path = "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing/meningioma/Te-aug-me_100.jpg"

img = image.load_img(
    img_path,
    target_size=(224,224)
)

img_array = image.img_to_array(img)

img_array = np.expand_dims(img_array, axis=0)

img_array = img_array / 255.0

prediction = model.predict(img_array)

predicted_class = np.argmax(prediction)

# New Class Names
class_names = {
    0: "Glioma",
    1: "Meningioma",
    2: "No Tumor",
    3: "Pituitary"
}

print(
    "Predicted Class:",
    class_names[predicted_class]
)

In [ ]:
!pip install streamlit

In [ ]:
mobile_model.save("best_brain_tumor_model.keras")

In [ ]:
%%writefile app.py
import streamlit as st
import numpy as np

from PIL import Image

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
# =========================
# FILE UPLOADER
# =========================

uploaded_file = st.file_uploader(
    "Upload MRI Image",
    type=["jpg", "jpeg", "png"]
)


# =========================
# PREDICTION
# =========================



    if uploaded_file is not None:

    img = Image.open(uploaded_file).convert("RGB")

    st.image(
        img,
        caption="Uploaded MRI Image",
        use_container_width=True
    )

    img = img.resize((224,224))

    img = img.convert("RGB")

    img_array = image.img_to_array(img)

    img_array = np.expand_dims(img_array, axis=0)

    img_array = img_array / 255.0

    prediction = model.predict(img_array)

    predicted_class = np.argmax(prediction)

    confidence = np.max(prediction) * 100

    st.success(
        f"Prediction: {class_names[predicted_class]}"
    )

    st.info(
        f"Confidence: {confidence:.2f}%"
    )

    # probabilities
    st.subheader("Prediction Probabilities")

    for i in range(len(class_names)):

        st.write(
            f"{class_names[i]} : {prediction[0][i]*100:.2f}%"
        )